In [ ]:
!pip install -q mediapipe gradio gtts jiwer transformers accelerate bitsandbytes

In [ ]:
!pip install -q mediapipe==0.10.14

In [1]:
import mediapipe as mp

mp_holistic = mp.solutions.holistic 

2026-04-19 08:33:28.233239: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776587608.710804     117 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776587608.820172     117 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776587609.966214     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776587609.966253     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776587609.966257     117 computation_placer.cc:177] computation placer alr

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow memory growth enabled.")
    except RuntimeError as e:
        print(f"TF Memory Error: {e}") 

TensorFlow memory growth enabled.


In [3]:
import os
import time
import json
import numpy as np
import tensorflow as tf
import mediapipe as mp
import cv2
import gradio as gr
from gtts import gTTS
from transformers import pipeline
from huggingface_hub import login

print(f'Tensorflow V{tf.__version__}')

# Configurations
INPUT_SIZE = 128 
N_DIMS = 3
NUM_CLASSES = 250
MODEL_WEIGHTS_PATH = '/kaggle/input/models/idowuadamo/hybrid-model-toptier/tensorflow2/default/1/hybrid_model_abs.weights.h5'

# Landmark indices
LIPS_IDXS0 = np.array([
        61, 185, 40, 39, 37, 0, 267, 269, 270, 409,
        291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
        95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    ])
LEFT_HAND_IDXS0 = np.arange(468,489)
RIGHT_HAND_IDXS0 = np.arange(522,543)
LEFT_POSE_IDXS0 = np.array([502, 504, 506, 508, 510])
RIGHT_POSE_IDXS0 = np.array([503, 505, 507, 509, 511])

LANDMARK_IDXS_LEFT_DOMINANT0 = np.concatenate((LIPS_IDXS0, LEFT_HAND_IDXS0, LEFT_POSE_IDXS0))
LANDMARK_IDXS_RIGHT_DOMINANT0 = np.concatenate((LIPS_IDXS0, RIGHT_HAND_IDXS0, RIGHT_POSE_IDXS0))
HAND_IDXS0 = np.concatenate((LEFT_HAND_IDXS0, RIGHT_HAND_IDXS0), axis=0)
N_COLS = LANDMARK_IDXS_LEFT_DOMINANT0.size

LIPS_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LIPS_IDXS0)).squeeze()
LEFT_HAND_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LEFT_HAND_IDXS0)).squeeze()
RIGHT_HAND_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, RIGHT_HAND_IDXS0)).squeeze()
POSE_IDXS = np.argwhere(np.isin(LANDMARK_IDXS_LEFT_DOMINANT0, LEFT_POSE_IDXS0)).squeeze()

N_COLS_FINAL = N_COLS * 9 

try:
    json_path = "/kaggle/input/competitions/asl-signs/sign_to_prediction_index_map.json" #"/kaggle/input/asl-signs/sign_to_prediction_index_map.json"
    with open(json_path, 'r') as f:
        data = json.load(f)
        ORD2SIGN = {v: k for k, v in data.items()}
        print("Label map loaded successfully.")
except Exception as e:
    print(f"Could not load map from {json_path}. Error: {e}")
    ORD2SIGN = {i: f"sign_{i}" for i in range(NUM_CLASSES)}

Tensorflow V2.19.0
Label map loaded successfully.


In [5]:
class PreprocessLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(PreprocessLayer, self).__init__() 
        self.lips_idxs = LIPS_IDXS
        self.left_hand_idxs = LEFT_HAND_IDXS
        self.pose_idxs = POSE_IDXS
        self.landmark_idxs_left = LANDMARK_IDXS_LEFT_DOMINANT0
        self.landmark_idxs_right = LANDMARK_IDXS_RIGHT_DOMINANT0
        
    @tf.function
    def call(self, data0):
        data0 = tf.cast(data0, tf.float32)

        # Filter dominant hand 
        left_hand_sum = tf.math.reduce_sum(tf.where(tf.math.is_nan(tf.gather(data0, LEFT_HAND_IDXS0, axis=1)), 0.0, 1.0))
        right_hand_sum = tf.math.reduce_sum(tf.where(tf.math.is_nan(tf.gather(data0, RIGHT_HAND_IDXS0, axis=1)), 0.0, 1.0))
        left_dominant = left_hand_sum >= right_hand_sum
        
        if left_dominant:
            frames_hands_non_nan_sum = tf.math.reduce_sum(
                tf.where(tf.math.is_nan(tf.gather(data0, LEFT_HAND_IDXS0, axis=1)), 0.0, 1.0), axis=[1, 2]
            )
            data = tf.gather(data0, self.landmark_idxs_left, axis=1)
        else:
            frames_hands_non_nan_sum = tf.math.reduce_sum(
                tf.where(tf.math.is_nan(tf.gather(data0, RIGHT_HAND_IDXS0, axis=1)), 0.0, 1.0), axis=[1, 2]
            )
            data = tf.gather(data0, self.landmark_idxs_right, axis=1)
            # Mirror X coordinate for right-handers
            data = tf.concat([
                -1.0 * tf.expand_dims(data[:, :, 0], axis=-1),
                tf.expand_dims(data[:, :, 1], axis=-1),
                tf.expand_dims(data[:, :, 2], axis=-1)
            ], axis=-1)
            
        non_empty_frames_idxs = tf.where(frames_hands_non_nan_sum > 0)
        non_empty_frames_idxs = tf.squeeze(non_empty_frames_idxs, axis=1)
        data = tf.gather(data, non_empty_frames_idxs, axis=0)

        # Normalization (Anchor to Lips/Nose)
        lips = data[:, :40, :] 
        lips_mean = tf.math.reduce_mean(tf.where(tf.math.is_nan(lips), 0.0, lips), axis=1, keepdims=True)
        lips_std = tf.math.reduce_std(tf.where(tf.math.is_nan(data), 0.0, data), axis=[1,2], keepdims=True) + 1e-6
        data = (data - lips_mean) / lips_std
        data = tf.where(tf.math.is_nan(data), 0.0, data)

        # Resizing / Interpolation
        N_FRAMES = tf.shape(data)[0]
        if N_FRAMES < INPUT_SIZE:
            non_empty_frames_idxs = tf.pad(
                tf.cast(non_empty_frames_idxs, tf.float32), 
                [[0, INPUT_SIZE - N_FRAMES]], constant_values=-1
            )
            data = tf.pad(data, [[0, INPUT_SIZE - N_FRAMES], [0,0], [0,0]], constant_values=0)
        else:
            data_flat = tf.reshape(data, [1, N_FRAMES, -1, 1])
            data_resized = tf.image.resize(
                data_flat, [INPUT_SIZE, tf.shape(data_flat)[2]], method=tf.image.ResizeMethod.BILINEAR
            )
            data = tf.reshape(data_resized, [INPUT_SIZE, -1, N_DIMS])
            non_empty_frames_idxs = tf.linspace(0.0, tf.cast(N_FRAMES, tf.float32), INPUT_SIZE)

        # Motion Features
        dx = data[1:, :, :] - data[:-1, :, :]
        dx = tf.concat([tf.zeros_like(data[:1, :, :]), dx], axis=0)
        
        ddx = dx[1:, :, :] - dx[:-1, :, :]
        ddx = tf.concat([tf.zeros_like(dx[:1, :, :]), ddx], axis=0)
        
        data = tf.concat([data, dx, ddx], axis=-1)
        data = tf.reshape(data, (INPUT_SIZE, -1))
        
        return data, non_empty_frames_idxs

preprocess_layer = PreprocessLayer()

In [6]:
# Custom Layers
class EcaLayer(tf.keras.layers.Layer):
    def __init__(self, kernel_size=5, **kwargs):
        super().__init__(**kwargs)
        self.conv = tf.keras.layers.Conv1D(1, kernel_size=kernel_size, padding='same', use_bias=False)

    def call(self, x):
        attn = tf.reduce_mean(x, axis=1, keepdims=True)
        attn = tf.transpose(attn, (0, 2, 1))
        attn = self.conv(attn)
        attn = tf.transpose(attn, (0, 2, 1))
        return x * tf.math.sigmoid(attn)

class Conv1DBlock(tf.keras.layers.Layer):
    def __init__(self, dim, kernel_size=11, drop_rate=0.2, expand=4):
        super().__init__()
        self.conv = tf.keras.layers.DepthwiseConv1D(kernel_size, padding='same', use_bias=False)
        self.bn = tf.keras.layers.BatchNormalization()
        self.act = tf.keras.layers.Activation('swish')
        self.se = EcaLayer(kernel_size=5)
        self.project = tf.keras.layers.Dense(dim, use_bias=False)
        self.drop = tf.keras.layers.Dropout(drop_rate)
        
    def call(self, x, training=None):
        skip = x
        x = self.conv(x)
        x = self.bn(x, training=training)
        x = self.act(x)
        x = self.se(x)
        x = self.project(x)
        if training: x = self.drop(x)
        return x + skip

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="gelu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output) 

class LearnablePositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.pos_embedding = tf.keras.layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        max_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=max_len, delta=1)
        return x + self.pos_embedding(positions)

class MaskingLayer(tf.keras.layers.Layer):
    def call(self, inputs):
        frames, non_empty_frame_idxs = inputs
        mask = tf.math.not_equal(non_empty_frame_idxs, -1)
        mask = tf.cast(mask, dtype=frames.dtype)
        return frames * tf.expand_dims(mask, -1)

def build_inference_model():
    embed_dim = 192
    num_heads = 4
    ff_dim = embed_dim * 2
    
    frames = tf.keras.layers.Input([INPUT_SIZE, N_COLS_FINAL], dtype=tf.float16, name='frames')
    non_empty_frame_idxs = tf.keras.layers.Input([INPUT_SIZE], dtype=tf.float16, name='non_empty_frame_idxs')
    
    x = MaskingLayer(name='input_masking')([frames, non_empty_frame_idxs])
    x = tf.keras.layers.Dense(embed_dim, use_bias=False, name='stem_conv')(x)
    x = tf.keras.layers.BatchNormalization(momentum=0.95, name='stem_bn')(x)
    
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    
    x = LearnablePositionalEmbedding(INPUT_SIZE, embed_dim)(x)
    x = TransformerBlock(embed_dim, num_heads, ff_dim, rate=0.2)(x)
    
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    x = Conv1DBlock(embed_dim, kernel_size=17, drop_rate=0.2)(x)
    
    x = TransformerBlock(embed_dim, num_heads, ff_dim, rate=0.2)(x)
    
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(0.8)(x) 
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='classifier')(x)
    
    return tf.keras.models.Model(inputs=[frames, non_empty_frame_idxs], outputs=outputs)

# Initialize and Load Weights
tf.keras.backend.clear_session()
model = build_inference_model()
print("Model initialized. Loading weights...")

try:
    model.load_weights(MODEL_WEIGHTS_PATH)
    print(f"Weights successfully loaded from:\n{MODEL_WEIGHTS_PATH}")
except Exception as e:
    print(f"Failed to load weights. Please verify the path.\nError: {e}") 

I0000 00:00:1776587759.444050     117 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776587759.450047     117 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model initialized. Loading weights...
Weights successfully loaded from:
/kaggle/input/models/idowuadamo/hybrid-model-toptier/tensorflow2/default/1/hybrid_model_abs.weights.h5


In [7]:
# Initialize LLM
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except:
    if os.environ.get("HF_TOKEN"): login(token=os.environ.get("HF_TOKEN"))

print("Loading Llama 3.2 1B Instruct...")
import torch
from transformers import pipeline

llm = pipeline(
    "text-generation", 
    model="meta-llama/Llama-3.2-1B-Instruct", 
    device_map="auto", 
    torch_dtype=torch.float16,
    model_kwargs={"attn_implementation": "eager"} 
)
print("LLM Loaded successfully.")

# Initialize MediaPipe Holistic
import mediapipe as mp
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
print("MediaPipe initialized.")

Loading Llama 3.2 1B Instruct...


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

LLM Loaded successfully.
MediaPipe initialized.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776587783.000400     751 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776587783.037187     751 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776587783.038386     749 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776587783.038730     751 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776587783.039846     748 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776587783.050049     

In [ ]:
def extract_landmarks(results):
    """Extract 543 landmarks per frame (Lips, Left Hand, Pose, Right Hand)."""
    landmarks = []
    
    if results.face_landmarks:
        landmarks.extend([[lm.x, lm.y, lm.z] for lm in results.face_landmarks.landmark])
    else:
        landmarks.extend([[float('nan')] * 3] * 468)

    if results.left_hand_landmarks:
        landmarks.extend([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark])
    else:
        landmarks.extend([[float('nan')] * 3] * 21)

    if results.pose_landmarks:
        landmarks.extend([[lm.x, lm.y, lm.z] for lm in results.pose_landmarks.landmark])
    else:
        landmarks.extend([[float('nan')] * 3] * 33)

    if results.right_hand_landmarks:
        landmarks.extend([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark])
    else:
        landmarks.extend([[float('nan')] * 3] * 21)

    return np.array(landmarks, dtype=np.float32)

def refine_to_sentence(sign_sequence):
    """Converts ASL glosses to English using Llama 3.2."""
    if not sign_sequence:
        return "No signs detected."
        
    glosses_str = " ".join(sign_sequence)
    messages = [
        {"role": "system", "content": "You are a fast, accurate ASL-to-English translation engine. Convert the provided ASL gloss sequence into a fluent, grammatically correct English sentence. Do not add explanations. Output ONLY the final translated sentence."},
        {"role": "user", "content": f"Glosses: {glosses_str}"}
    ]
    
    prompt = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm(prompt, max_new_tokens=64, do_sample=False, return_full_text=False)[0]['generated_text']
    return out.strip()

def process_video_pipeline(video_path):
    """End-to-End inference processing."""
    start_time = time.time()
    if not video_path:
        return "No video provided.", "", "N/A", None, "0.0s / 0.0 FPS"

    # Video Parsing & MediaPipe
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_lms = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(frame_rgb)
        frames_lms.append(extract_landmarks(results))
    cap.release()
    
    if len(frames_lms) < 10:
        return "Video too short.", "Video too short.", "N/A", None, "0.0s"
        
    mp_time = time.time()
    
    # Tensor Batching & Preprocessing
    data = np.array(frames_lms, dtype=np.float32)
    batch_frames, batch_idxs = [], []
    window_step = 32  
    
    for start in range(0, max(1, len(data) - INPUT_SIZE + 1), window_step):
        window = data[start:start + INPUT_SIZE]
        if len(window) < INPUT_SIZE:
            pad_len = INPUT_SIZE - len(window)
            window = np.concatenate((window, np.zeros((pad_len, 543, 3))), axis=0)
            
        p_data, n_idxs = preprocess_layer(window)
        batch_frames.append(p_data)
        batch_idxs.append(n_idxs)
        
    prep_time = time.time()
        
    # TF GPU Inference
    # Convert lists to dense tensors for prediction
    preds_raw = model.predict({
        'frames': tf.convert_to_tensor(batch_frames, dtype=tf.float16), 
        'non_empty_frame_idxs': tf.convert_to_tensor(batch_idxs, dtype=tf.float16)
    }, batch_size=32, verbose=0)
    
    # Gloss Aggregation
    CONFIDENCE_THRESH = 0.4
    unique_seq = []
    conf_log = []
    
    for p in preds_raw:
        pred_idx = p.argmax()
        conf = p[pred_idx]
        sign = ORD2SIGN.get(pred_idx, 'unknown')
        
        if sign != "unknown" and conf > CONFIDENCE_THRESH:
            if not unique_seq or sign != unique_seq[-1]:
                unique_seq.append(sign)
                conf_log.append(f"{sign} ({conf:.1%})")
                
    inf_time = time.time()
    
    if not unique_seq:
        return "No confident signs recognized.", "None", "N/A", None, f"Processed in {inf_time - start_time:.2f}s"
        
    raw_glosses = " ".join(unique_seq)
    formatted_confs = " | ".join(conf_log)
    
    # LLM Refinement
    refined_sentence = refine_to_sentence(unique_seq)
    llm_time = time.time()
    
    # Text-to-Speech
    audio_path = 'output.mp3'
    try:
        tts = gTTS(refined_sentence, lang='en')
        tts.save(audio_path)
    except:
        audio_path = None
        
    end_time = time.time()
    
    total_time = end_time - start_time
    process_fps = total_frames / total_time if total_time > 0 else 0
    stats = (f"Total Latency: {total_time:.2f}s | "
             f"Speed: {process_fps:.1f} FPS\n"
             f"(Breakdown - MediaPipe: {mp_time-start_time:.2f}s, TF: {inf_time-prep_time:.2f}s, LLM: {llm_time-inf_time:.2f}s)")
             
    return raw_glosses, formatted_confs, refined_sentence, audio_path, stats

# Gradio UI
custom_css = """
#large_text textarea { font-size: 28px !important; font-weight: 700; color: #1a202c; }
"""
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=custom_css) as demo:
    gr.Markdown("# Real-Time ASL Translation Pipeline")
    
    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### 1. Video Input")
            vid_input = gr.Video(sources=["webcam", "upload"], label="Record or Upload", include_audio=False)
            btn_translate = gr.Button("🚀 Run Full Inference Pipeline", variant="primary", size="lg")
            
            gr.Markdown("### 2. Pipeline Diagnostics")
            out_stats = gr.Textbox(label="Processing Time & FPS", interactive=False, lines=2)
            out_confs = gr.Textbox(label="Per-Gloss Confidence Scores", interactive=False)
            
        with gr.Column(scale=5):
            gr.Markdown("### 3. Translation Output")
            out_sentence = gr.Textbox(label="Final English Sentence", elem_id="large_text", lines=3, interactive=False)
            out_audio = gr.Audio(label="Spoken Translation", autoplay=True)
            out_glosses = gr.Textbox(label="Raw Gloss Sequence (Tier 1)", interactive=False)

    btn_translate.click(
        fn=process_video_pipeline, inputs=vid_input,
        outputs=[out_glosses, out_confs, out_sentence, out_audio, out_stats]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_117/3162858124.py:139: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=custom_css) as demo:
/tmp/ipykernel_117/3162858124.py:139: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=custom_css) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2461376ecc1ded9b25.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
